In [1]:
import polars as pl

In [2]:
electricity_path = "C:/Users/letung373/Desktop/dmp-project/data/raw/data/meters/cleaned/electricity_cleaned.csv"
metadata_path = "C:/Users/letung373/Desktop/dmp-project/data/raw/data/metadata/metadata.csv"

In [ ]:
name_id = ["Bear*", "Bull*", "Bobcat*"]

In [4]:
# 1. Đọc header để lấy danh sách toàn bộ cột building_id (không load dữ liệu)
all_cols = pl.read_csv(electricity_path, n_rows=0).columns

# 2. Khớp tiền tố: "Bear*" -> startswith("Bear")
prefixes = [p.removesuffix("*") for p in name_id]          # ["Bear", "Bull", "Bobcat"]
buildings = [c for c in all_cols if any(c.startswith(p) for p in prefixes)]

print(f"Tổng số tòa khớp: {len(buildings)}")
for p in prefixes:
    cnt = sum(1 for c in buildings if c.startswith(p))
    print(f"  {p}* : {cnt} tòa")

Tổng số tòa khớp: 1579
  B* : 250 tòa
  e* : 0 tòa
  a* : 0 tòa
  r* : 0 tòa
  * : 1579 tòa
  ,* : 0 tòa
   * : 0 tòa
  B* : 250 tòa
  u* : 0 tòa
  l* : 0 tòa
  l* : 0 tòa
  * : 1579 tòa
  ,* : 0 tòa
   * : 0 tòa
  B* : 250 tòa
  o* : 0 tòa
  b* : 0 tòa
  c* : 0 tòa
  a* : 0 tòa
  t* : 1 tòa
  * : 1579 tòa


In [ ]:
# 3. Chỉ đọc các cột cần thiết (tiết kiệm bộ nhớ) và tính missing rate (%)
df = pl.read_csv(electricity_path, columns=["timestamp", *buildings], try_parse_dates=True)

n = df.height
missing = (
    pl.DataFrame({
        "building_id": buildings,
        "missing_count": [df[b].null_count() for b in buildings],
    })
    .with_columns(
        total_rows=pl.lit(n),
        # missing_rate tính theo % (0-100)
        missing_rate=(pl.col("missing_count") / n * 100).round(2),
    )
    .sort("missing_rate", descending=True)
)

print(f"Tổng số dòng (timestamps): {n}")
missing

In [ ]:
# 4. Tổng hợp missing rate theo tiền tố (Bear / Bull / Bobcat)
summary = (
    missing.with_columns(
        prefix=pl.col("building_id").str.split("_").list.first()
    )
    .group_by("prefix")
    .agg(
        n_buildings=pl.len(),
        mean_missing_rate=pl.col("missing_rate").mean().round(4),
        median_missing_rate=pl.col("missing_rate").median().round(4),
        max_missing_rate=pl.col("missing_rate").max().round(4),
    )
    .sort("mean_missing_rate", descending=True)
)
summary

In [ ]:
# 5. Xuất ra CSV (missing_rate tính theo %)
out_path = "C:/Users/letung373/Desktop/dmp-project/notebooks/forecasting/EDA/missing_rate.csv"

export = (
    missing.with_columns(prefix=pl.col("building_id").str.split("_").list.first())
    .select("prefix", "building_id", "missing_count", "total_rows", "missing_rate")
)

export.write_csv(out_path)
print(f"Đã ghi {export.height} dòng ra: {out_path}")
export.head(10)

In [6]:

import polars as pl

electricity_path = "C:/Users/letung373/Desktop/dmp-project/data/raw/data/meters/cleaned/electricity_cleaned.csv"
name_id = ["Bear*", "Bull*", "Bobcat*"]
out_path = "C:/Users/letung373/Desktop/dmp-project/notebooks/forecasting/EDA/missing_rate.csv"

all_cols = pl.read_csv(electricity_path, n_rows=0).columns
prefixes = [p.removesuffix("*") for p in name_id]
buildings = [c for c in all_cols if any(c.startswith(p) for p in prefixes)]

df = pl.read_csv(electricity_path, columns=["timestamp", *buildings], try_parse_dates=True)
n = df.height

missing = (
    pl.DataFrame({
        "building_id": buildings,
        "missing_count": [df[b].null_count() for b in buildings],
    })
    .with_columns(
        total_rows=pl.lit(n),
        missing_rate=(pl.col("missing_count") / n * 100).round(2),
    )
    .with_columns(prefix=pl.col("building_id").str.split("_").list.first())
    .select("prefix", "building_id", "missing_count", "total_rows", "missing_rate")
    .sort("missing_rate", descending=True)
)

missing.write_csv(out_path)
print(f"Matched buildings: {len(buildings)} | rows: {n}")
print(f"Wrote -> {out_path}")
print(missing.head(10))


Matched buildings: 250 | rows: 17544
Wrote -> C:/Users/letung373/Desktop/dmp-project/notebooks/forecasting/EDA/missing_rate.csv
shape: (10, 5)
┌────────┬──────────────────────────┬───────────────┬────────────┬──────────────┐
│ prefix ┆ building_id              ┆ missing_count ┆ total_rows ┆ missing_rate │
│ ---    ┆ ---                      ┆ ---           ┆ ---        ┆ ---          │
│ str    ┆ str                      ┆ i64           ┆ i32        ┆ f64          │
╞════════╪══════════════════════════╪═══════════════╪════════════╪══════════════╡
│ Bobcat ┆ Bobcat_education_Barbra  ┆ 17434         ┆ 17544      ┆ 99.37        │
│ Bobcat ┆ Bobcat_warehouse_Charlie ┆ 16982         ┆ 17544      ┆ 96.8         │
│ Bobcat ┆ Bobcat_education_Seth    ┆ 16885         ┆ 17544      ┆ 96.24        │
│ Bobcat ┆ Bobcat_public_Angie      ┆ 14563         ┆ 17544      ┆ 83.01        │
│ Bobcat ┆ Bobcat_education_Emile   ┆ 14509         ┆ 17544      ┆ 82.7         │
│ Bobcat ┆ Bobcat_education_Rodrick ┆